# Run Build AI Agents Free on Android—without Termux

This notebook runs the repository on Google Colab's hosted Python runtime. Your Android phone only needs a browser. Run each cell in order with the **play** button.

## 1. Clone or refresh the repository

A Colab runtime is temporary. If Google resets it, start again from this cell.

In [ ]:
from pathlib import Path
import os
import subprocess

REPOSITORY = "https://github.com/6900-sudo/build-ai-agents-free.git"
PROJECT = Path("/content/build-ai-agents-free")

if PROJECT.exists():
    subprocess.run(["git", "-C", str(PROJECT), "pull", "--ff-only"], check=True)
else:
    subprocess.run(["git", "clone", REPOSITORY, str(PROJECT)], check=True)

os.chdir(PROJECT)
print("Ready in", PROJECT)

## 2. Install the Python packages

The extra `ddgs` package keeps DuckDuckGo search compatible with current LangChain integrations.

In [ ]:
%pip install -q -r requirements.txt ddgs

## 3. Add API keys safely

1. Create a Groq key at <https://console.groq.com/keys>.
2. In Colab, tap the **key** icon in the left sidebar to open **Secrets**.
3. Add a secret named `GROQ_API_KEY`, paste the value, and enable notebook access.
4. Optional: create a Gemini key at <https://aistudio.google.com/app/apikey> and add it as `GOOGLE_API_KEY` for the complete agent's fallback.

Never type a real key directly into a code cell, notebook output, chat, or GitHub file.

In [ ]:
from google.colab import userdata
import os

def read_secret(name, required=False):
    try:
        value = userdata.get(name)
    except Exception:
        value = None
    if required and not value:
        raise RuntimeError(f"Add {name} in Colab Secrets and enable notebook access.")
    return value

os.environ["GROQ_API_KEY"] = read_secret("GROQ_API_KEY", required=True)
google_key = read_secret("GOOGLE_API_KEY")
if google_key:
    os.environ["GOOGLE_API_KEY"] = google_key

print("Groq is configured; Gemini fallback is", "configured." if google_key else "not configured.")

## 4. Run the first agent

This confirms that Python, LangChain and Groq are working.

In [ ]:
!python src/01_first_agent.py

## 5. Run the complete agent

The complete example adds DuckDuckGo search, word count, conversation memory and Groq-to-Gemini fallback. Add both secrets first so fallback remains available if Groq is rate-limited.

In [ ]:
import subprocess

if not google_key:
    print("Add GOOGLE_API_KEY in Colab Secrets before running the fallback example.")
else:
    subprocess.run(["python", "src/agent.py"], check=True)

## Troubleshooting

- **Secret not found:** check the exact uppercase name and turn on notebook access beside it.
- **Rate limit:** wait briefly or configure the Gemini fallback. Free-tier limits can change.
- **DuckDuckGo error:** wait a few seconds and rerun; its free search endpoint rate-limits bursts.
- **Runtime disconnected:** reconnect, then run all cells from the top. Colab's temporary files and environment variables disappear when the runtime resets.
- **Privacy:** prompts go to the selected AI/search provider. Do not use free tiers for confidential data.